# PyGeoFetch — 07: Copernicus, USGS & Authenticated Providers

Deep dive into authenticated providers, Sentinel-1C/D SLC products,
and orbit file management for InSAR.

### What you'll learn
- Copernicus CDSE: Sentinel-1/2/3/5P (including SLC and S1C)
- USGS Earth Explorer: Landsat 1-9
- NASA Earthdata: MODIS, ICESat-2, GEDI
- Alaska SAR Facility: SLC products for InSAR
- Orbit file pre-download (Capability 3)
- Sentinel-1C constellation (Capability 2)

In [1]:

from pygeofetch import PyGeoFetch
from pygeofetch.core.authenticator import AuthManager
from pygeofetch.models.search_query import BoundingBox, SearchQuery

client = PyGeoFetch(log_level="WARNING")
auth   = AuthManager()
print("Checking stored credentials...")
entries = auth.list()
stored  = {e["provider"] for e in entries}
print(f"Providers with credentials: {sorted(stored) or 'none'}")

Checking stored credentials...
Providers with credentials: ['OpenTopography', 'copernicus', 'maxar_gbdx', 'opentopography', 'planet', 'sentinel_hub', 'usgs']


## 1. Copernicus — Sentinel-1C/D + SLC Support

In [ ]:
# Copernicus is the primary provider for:
# - Sentinel-1C/D GRD and SLC
# - Sentinel-2 L1C and L2A
# - Sentinel-3 and Sentinel-5P

COPERNICUS_READY = "copernicus" in stored
print(f"Copernicus credentials: {'✓ ready' if COPERNICUS_READY else '✗ not configured'}")

if not COPERNICUS_READY:
    print()
    print("To configure:")
    print("  client.add_credentials('copernicus', username='email@example.com', password='PASS')")
    print("  # or register free at: https://dataspace.copernicus.eu/")
    print()
    print("Environment variable alternative:")
    print("  PYGEOFETCH_COPERNICUS_USERNAME=email@example.com")
    print("  PYGEOFETCH_COPERNICUS_PASSWORD=PASS")

# Capability 1: product_type field + SLC routing
print()
print("Sentinel-1 product types available on Copernicus:")
products = [
    ("GRD",     "Ground Range Detected — intensity only, 800MB-2GB"),
    ("SLC",     "Single Look Complex — phase preserved, 4-8GB, required for InSAR"),
    ("GRD-COG", "GRD as Cloud Optimized GeoTIFF"),
    ("OCN",     "Ocean products (wind, wave, radial velocity)"),
]
for ptype, desc in products:
    print(f"  {ptype:<10}  {desc}")

In [ ]:
# ── Sentinel-1 SLC search via Copernicus ────────────────────────────────────
# SLC = Single Look Complex — phase information preserved, required for InSAR.
# GRD = Ground Range Detected — intensity only, smaller, all providers.

if COPERNICUS_READY:
    q_slc = SearchQuery(
        bbox=BoundingBox.from_string("-0.3,5.4,0.2,5.9"),  # Ghana/Accra area
        start_date="2026-06-01",
        end_date="2026-06-30",
        satellites=["Sentinel-1C"],   # S1C = active constellation (May 2025+)
        max_results=10,
        collections=["SENTINEL-1"],   # Copernicus OData collection name (note: plural)
    )
    q_slc.set_product_type("SLC")    # Adds productType='SLC' to the OData filter

    results_slc = client.search(q_slc, providers=["copernicus"])
    print(f"Found {len(results_slc)} Sentinel-1 SLC scenes")
    print()
    if results_slc:
        print(f"  {'Satellite':<15} {'Polarisation':<12} {'Pass':<13} {'Orbit':<7} Date")
        print("  " + "-" * 65)
        for r in results_slc[:5]:
            sat  = (r.satellite      or "—")[:14]
            pol  = (r.polarisation   or "—")[:11]
            pdir = (r.pass_direction or "—")[:12]
            orb  = str(r.relative_orbit or "—")[:6]
            date = str(r.datetime)[:10] if r.datetime else "—"
            print(f"  {sat:<15} {pol:<12} {pdir:<13} {orb:<7} {date}")
    else:
        print("  No SLC scenes in this date range.")
        print("  Try: start_date='2025-01-01', end_date='2025-12-31'")
else:
    print("Add Copernicus credentials first:")
    print("  client.add_credentials('copernicus', username='EMAIL', password='PASS')")

# ── SLC provider routing (for mixed-provider searches) ──────────────────────
print()
print("SLC routing — GRD-only providers removed automatically:")
mixed = ["planetary_computer", "aws_earth", "element84", "copernicus", "alaska_satellite_facility"]
routed = client._route_slc_providers(mixed)
print(f"  Requested : {mixed}")
print(f"  Routed    : {routed}")
print("  (planetary_computer, aws_earth, element84 removed — GRD only)")


## 2. Orbit Files for InSAR — Capability 3

In [ ]:
# Precise orbit files (POEORB) are required for mm-precision InSAR
# Published 21 days after acquisition by ESA
# Restituted orbits (RESORB) available within ~3 hours

print("engine.fetch_orbit_file() — full API:")
print()
print("  path = engine.fetch_orbit_file(")
print("      product_name='S1C_IW_SLC__1SDV_20260601T053000_...',")
print("      output_dir='./orbits/',")
print("      orbit_type='precise',   # or 'restituted'")
print("  )")
print()
print("Orbit file naming convention:")
print("  S1C_OPER_AUX_POEORB_OPOD_<proc_time>_V<validity_start>_<validity_stop>.EOF")
print()

# Demonstrate orbit logic offline
from pygeofetch.core.orbits import (
    _find_matching_orbit_file,
    _parse_acquisition_datetime,
    _parse_satellite,
)

product = "S1C_IW_SLC__1SDV_20260601T053000_20260601T053027_053442_067B1A_F3A2"
acq_dt  = _parse_acquisition_datetime(product)
sat     = _parse_satellite(product)
print(f"Product:  {product[:50]}...")
print(f"Acq DT:   {acq_dt}")
print(f"Satellite:{sat}")

# Simulate HTML listing
listing = '''
<a href="S1C_OPER_AUX_POEORB_OPOD_20260622T121000_V20260601T000000_20260603T000000.EOF">
<a href="S1C_OPER_AUX_POEORB_OPOD_20260623T110000_V20260603T000000_20260605T000000.EOF">
'''
match = _find_matching_orbit_file(listing, acq_dt)
print(f"Matched: {match}")

from datetime import datetime

days_since = (datetime.utcnow() - acq_dt).days
print()
if days_since < 21:
    print(f"Note: Only {days_since}d since acquisition — POEORB may not be published yet")
    print("      Use orbit_type='restituted' for near-real-time processing")
else:
    print(f"{days_since}d since acquisition — POEORB should be available")

## 3. USGS Earth Explorer — Landsat 1-9

In [ ]:
USGS_READY = "usgs" in stored
print(f"USGS credentials: {'✓ ready' if USGS_READY else '✗ not configured'}")
print()
print("Register free at: https://ers.cr.usgs.gov/register")
print()
print("Available datasets:")
datasets = [
    ("Landsat-9",  "30m",  "OLI-2/TIRS-2",        "16-day revisit (2021-)"),
    ("Landsat-8",  "30m",  "OLI/TIRS",             "16-day revisit (2013-)"),
    ("Landsat-7",  "30m",  "ETM+",                 "16-day (SLC-off 2003)"),
    ("Landsat-4/5","30m",  "TM",                   "1982-2013"),
    ("Landsat-1-3","79m",  "MSS",                  "1972-1983"),
    ("ASTER",      "15m",  "VNIR/SWIR/TIR",        "2000-"),
    ("MODIS",      "250m", "Terra/Aqua",            "daily"),
    ("SRTM",       "30m",  "DEM",                  "global 1-arc-second"),
]
print(f"  {'Dataset':<12} {'Res':<6} {'Sensor':<18} {'Notes'}")
print("-" * 60)
for ds in datasets:
    print(f"  {ds[0]:<12} {ds[1]:<6} {ds[2]:<18} {ds[3]}")

## 4. Alaska SAR Facility — Sentinel-1 SLC + InSAR

In [ ]:
ASF_READY = "alaska_satellite_facility" in stored
print(f"ASF credentials: {'✓ ready' if ASF_READY else '✗ not configured (uses NASA Earthdata login)'}")
print()
print("ASF DAAC hosts:")
print("  - Sentinel-1A/B/C/D  GRD and SLC")
print("  - ALOS PALSAR-1/2    L-band SAR")
print("  - ERS-1/2, JERS-1    legacy SAR")
print("  - UAVSAR             airborne SAR")
print()
print("SLC search via ASF:")
print("  q = SearchQuery(bbox=BoundingBox.from_string('...'))")
print("  q.set_product_type('SLC')")
print("  results = client.search(q, providers=['alaska_satellite_facility'])")
print()
print("SLC routing: ASF is listed in _SLC_CAPABLE — no rerouting needed")
print("  _SLC_CAPABLE = {copernicus, alaska_satellite_facility, asf_vertex, eodag}")

## 5. Provider Authentication Reference

In [ ]:
print("Authentication reference for all 12 authenticated providers:")
print()
auth_table = [
    ("usgs",                    "username/password", "ers.cr.usgs.gov/register",     "free"),
    ("copernicus",              "username/password", "dataspace.copernicus.eu",       "free"),
    ("nasa_earthdata",          "username/password", "urs.earthdata.nasa.gov",        "free"),
    ("nasa_earthdata_cloud",    "username/password", "urs.earthdata.nasa.gov",        "free"),
    ("alaska_satellite_facility","username/password","urs.earthdata.nasa.gov",        "free"),
    ("opentopography",          "api_key",          "portal.opentopography.org",      "free tier"),
    ("planet",                  "api_key",          "planet.com/account",             "subscription"),
    ("sentinel_hub",            "client_id+secret", "apps.sentinel-hub.com",          "freemium"),
    ("maxar_gbdx",              "token",            "developers.maxar.com",           "subscription"),
    ("airbus_oneatlas",         "api_key",          "oneatlas.airbus.com",            "subscription"),
    ("google_earth_engine",     "service_account",  "console.cloud.google.com",       "free tier"),
    ("terrabotics",             "api_key",          "terrabotics.earth",              "subscription"),
]
print(f"  {'Provider':<30} {'Auth Type':<20} {'Cost'}")
print("-" * 70)
for prov, atype, url, cost in auth_table:
    print(f"  {prov:<30} {atype:<20} {cost}")

---
## Summary
- Copernicus: Sentinel-1C/D GRD+SLC, Sentinel-2, Sentinel-3/5P
- SLC provider routing: planetary_computer/aws_earth/element84 auto-routed to copernicus
- Orbit file management: fetch_orbit_file() with caching, POEORB/RESORB types
- USGS: Landsat 1-9, ASTER, MODIS, SRTM (free with registration)
- ASF: SLC products for InSAR (Sentinel-1A/B/C/D, ALOS PALSAR)
- All 12 authenticated providers with authentication reference

**Next:** Notebook 08 — CLI Complete Reference